# Model retraining with the ML App client

This notebook demonstrates dataset version upload and retraining without manually constructing API requests or resolving resource IDs. The client streams the file, selects the latest published pipeline version, and polls bounded run metadata.

In [ ]:
BUSINESS_CASE_NAME = "Estates Sell Prices"
DATASET_NAME = "sale-prices"
PIPELINE_NAME = "Estates Sell Prices - AutoFEML"
INPUT_FILE = None  # For example r"C:\data\sale-prices.csv"; None generates 30k demo records

## Client and authentication

Install the client from the repository root with `pip install -e .`. In automation, provide `ML_APP_API_URL` and `ML_APP_ACCESS_TOKEN` through a secret manager. For local interactive use, the notebook prompts for credentials when no token is configured.

In [ ]:
from pathlib import Path

from ml_app_client import MLAppClient

client = MLAppClient.connect()

## Input file

The generator is used for demonstration only and does not make platform requests.

In [ ]:
if INPUT_FILE:
    training_file = Path(INPUT_FILE)
else:
    import sys

    helpers = next(
        path
        for path in [Path.cwd(), Path.cwd() / "examples" / "API-usage"]
        if (path / "api_example_helpers.py").is_file()
    )
    sys.path.insert(0, str(helpers))
    from api_example_helpers import build_training_file

    training_file = build_training_file()
training_file

## Upload the next dataset version

The client resolves the dataset attached to the Business Case and streams the file as a new immutable version.

In [ ]:
uploaded = client.upload_dataset_version(
    training_file,
    business_case_name=BUSINESS_CASE_NAME,
    dataset_name=DATASET_NAME,
)
rows = "unknown" if uploaded.row_count is None else f"{uploaded.row_count:,}"
print(f"Uploaded {uploaded.name} v{uploaded.version_number} ({rows} rows)")

## Start the pipeline

The client selects the latest published pipeline version. An input configured with `latest` is resolved by the backend to the version uploaded above.

In [ ]:
run = client.run_pipeline_by_name(
    business_case_name=BUSINESS_CASE_NAME,
    pipeline_name=PIPELINE_NAME,
)
print(f"Started run_id={run.id}; status={run.status}")

## Wait for completion

Polling retrieves bounded run metadata only. The full dataset is not transferred to the notebook, and a timeout does not cancel the platform job.

In [ ]:
finished = client.wait_for_pipeline_run(
    run,
    timeout=3600,
    on_update=lambda current: print(
        f"status={current.status}; processed_rows={current.processed_row_count}"
    ),
)
rows = (
    "unknown"
    if finished.processed_row_count is None
    else f"{finished.processed_row_count:,}"
)
print(f"Completed: {rows} processed rows")